# Fine-Tuning BERT for SMS Spam Detection

This notebook demonstrates how to fine-tune a pretrained **BERT (bert-base-uncased)** model for binary sequence classification to detect SMS spam. 

- **HAM (Normal Message)** = `0`
- **SPAM (Spam Message)** = `1`

Training took approximately **73 minutes** to run over 3 epochs, achieving **99.37%** validation accuracy.

## 1. Import Libraries
We load Python packages for data processing (`pandas`, `numpy`), machine learning (`scikit-learn`), deep learning (`torch`), and Hugging Face's `transformers`.

In [5]:
import pandas as pd 
import numpy as np 

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import torch
from torch.utils.data import Dataset

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments
)

## 2. Load and Preprocess Dataset
We load the `spam.csv` file using standard `ISO-8859-1` encoding, drop unused columns, rename the columns to `label` and `text`, and map label text (`ham` and `spam`) to binary integers (`0` and `1`).

In [6]:
df = pd.read_csv(r"C:\Users\harsh\OneDrive\Desktop\New folder (2)\dataset\spam.csv",
    encoding="ISO-8859-1")

df = df[['v1','v2']]
df.columns = ["label","text"]

df["label"] = df["label"].map({
    "ham": 0,
    "spam": 1
})

print(df.head())

   label                                               text
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro...


## 3. Analyze Dataset Distribution
Check dataset shape, check for nulls, and inspect class balance. Notice that the dataset is highly imbalanced toward HAM (normal messages).

In [7]:
print("Dataset Shape: ", df.shape)
print("\nMissing values: \n", df.isnull().sum())
print("\nSpam Count: ")
print(df["label"].value_counts())

Dataset Shape:  (5572, 2)

Missing values: 
 label    0
text     0
dtype: int64

Spam Count: 
label
0    4825
1     747
Name: count, dtype: int64


## 4. Split Train/Test Splits
We perform an 80/20 train/test split. The configuration parameters (`test_size=0.2`, `random_state=42`) must be identically replicated when evaluating the saved model.

In [8]:
from sklearn.model_selection import train_test_split

train_texts, test_texts, train_labels, test_labels = train_test_split(
    df["text"],
    df["label"],
    test_size = 0.2,
    random_state = 42
)

print("Training Samples: ", len(train_texts))
print("Testing Samples: ", len(test_texts))

Training Samples:  4457
Testing Samples:  1115


## 5. Load Pretrained BERT Tokenizer
We load the `bert-base-uncased` tokenizer from Hugging Face, which parses text into subwords that match BERT's vocabulary.

In [9]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
print("Tokenizer Load Successfully")

Tokenizer Load Successfully


## 6. Tokenize Text Data
Convert train and test text messages into index-encoded tensors, setting `max_length=128` with padding and truncation.

In [10]:
train_encodings = tokenizer(
    train_texts.tolist(),
    truncation = True,
    padding = True,
    max_length = 128
)

test_encodings = tokenizer(
    test_texts.tolist(),
    truncation = True,
    padding = True,
    max_length = 128
)

print("Trainig messages: ", len(train_encodings["input_ids"]))
print("Testing messages: ", len(test_encodings["input_ids"]))

Trainig messages:  4457
Testing messages:  1115


## 7. Load Pretrained BERT Model
Load `BertForSequenceClassification` from `bert-base-uncased` with 2 labels. The reported unexpected/missing keys are typical and indicate that we are initializing classification heads that will be trained on downstream labels.

In [11]:
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels = 2
)

print("Model Loaded Successfully")

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model Loaded Successfully


## 8. Configure Training Arguments
Define batch sizes, epochs, logging steps, and training strategies. We evaluate and save models at the end of each epoch, reloading the best epoch weights at the end.

In [12]:
training_args = TrainingArguments(
    output_dir = "./results",
    num_train_epochs = 3,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size = 8,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    logging_steps = 100,
    load_best_model_at_end = True,
    report_to = "none"
)

## 9. Define Evaluation Metrics
Implement accuracy calculation using `accuracy_score` from `scikit-learn` for Trainer validation.

In [13]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(
        pred.predictions,
        axis = 1
    )
    accuracy = accuracy_score(
        labels,
        preds
    )
    return{
        "accuracy" : accuracy
    }

## 10. Create Custom PyTorch Dataset
Extend `torch.utils.data.Dataset` to format tokens and labels into the dict items required by Hugging Face's `Trainer` API.

In [14]:
import torch
from torch.utils.data import Dataset

class SpamDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

## 11. Instantiate Datasets

In [15]:
train_dataset = SpamDataset(
    train_encodings,
    train_labels
)

test_dataset = SpamDataset(
    test_encodings,
    test_labels
)

print(len(train_dataset))
print(len(test_dataset))

4457
1115


## 12. Initialize Trainer
Set up the Hugging Face `Trainer` wrapper class passing training arguments, model, datasets, and metrics computation logic.

In [16]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    compute_metrics = compute_metrics
)

## 13. Fine-Tune BERT Classifier

> [!WARNING]
> **DO NOT rerun the cell below!** Fine-tuning BERT has already completed and took approximately **73 minutes** to execute on CPU. Rerunning this cell will overwrite the fine-tuned model weights.

In [17]:
trainer.train()

c:\Users\harsh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.075905,0.052862,0.992825
2,0.027183,0.047095,0.989238
3,0.000251,0.040684,0.993722


TrainOutput(global_step=1674, training_loss=0.03736623267953595, metrics={'train_runtime': 4387.9208, 'train_samples_per_second': 3.047, 'train_steps_per_second': 0.382, 'total_flos': 879514480304640.0, 'train_loss': 0.03736623267953595, 'epoch': 3.0})

## 14. Evaluate on Test Dataset
Run trainer evaluation to get final test metrics. The test accuracy is **99.37%**.

In [18]:
results = trainer.evaluate()
print("Evaluation results: ", results)

c:\Users\harsh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy
0.000251,0.040684,3,0.993722


Evaluation results:  {'eval_loss': 0.04068361967802048, 'eval_accuracy': 0.9937219730941704}


## 15. Save Model and Tokenizer
Save the fine-tuned BERT model and configured tokenizer locally.

In [19]:
save_path = r"C:\Users\harsh\OneDrive\Desktop\New folder (2)\model\saved_model"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Spam Detection Model saved Successfully!")

Spam Detection Model saved Successfully!


## 16. Local Prediction Helper
Define a quick prediction helper function to run inference on single messages directly inside this training notebook.

In [20]:
def predict_spam(message):
    inputs = tokenizer(
        message,
        return_tensors = "pt",
        truncation = True,
        padding = True,
        max_length = 128
    )

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = torch.softmax(
        outputs.logits,
        dim = 1
    )

    predicted_class = torch.argmax(
        probabilities,
        dim = 1
    ).item()

    confidence = probabilities[0][predicted_class].item()

    if predicted_class == 1:
        prediction = "SPAM"
    else:
        prediction = "HAM"

    return prediction, confidence

In [21]:
message = "Congratulations! You have won 50000 rupees. Click here to claim your prize."
prediction, confidence = predict_spam(message)

print("Message    :", message)
print("Prediction :", prediction)
print("Confidence :", round(confidence * 100, 2), "%")

Message    : Congratulations! You have won 50000 rupees. Click here to claim your prize.
Prediction : SPAM
Confidence : 99.97 %


In [22]:
message = "Hey Harsh, are you coming to college tomorrow?"
prediction, confidence = predict_spam(message)

print("Message    :", message)
print("Prediction :", prediction)
print("Confidence :", round(confidence * 100, 2), "%")

Message    : Hey Harsh, are you coming to college tomorrow?
Prediction : HAM
Confidence : 99.99 %
